# 🏥 Tech Challenge Fase 3 — Fase 1: Dataset e Preprocessing

> **⚡ Modo Colab Free:** dataset reduzido (~110 exemplos) para evitar custo de GPU e RAM.

**Pipeline:**
```
1. Instalação de dependências
2. Download de subset pequeno do PubMedQA (100 exemplos)
3. Dados sintéticos hospitalares (10 protocolos fixos, sem GPU)
4. Anonimização com regex
5. Formatação Alpaca JSONL para fine-tuning
6. Validação e exportação
```

## ☁️ 0. Google Drive — Workspace Compartilhado
> Monta o Drive para que os arquivos gerados aqui (dataset, JSONL) fiquem persistentes e acessíveis pelo `02_finetuning.ipynb` na mesma sessão ou em abas separadas.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/tech-challenge-fase3'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print(f'✅ Drive montado. Pasta do projeto: {PROJECT_DIR}')
except ImportError:
    # Fora do Colab — usa o diretório local normalmente
    print('ℹ️  Ambiente local detectado. Usando diretório atual:', os.getcwd())

## 📦 1. Instalação de Dependências
> Apenas o essencial — sem modelos NLP pesados.

In [ ]:
!pip install datasets pandas matplotlib -q
print('✅ Dependências instaladas!')

## 📥 2. Download do PubMedQA — subset de 100 exemplos

In [ ]:
from datasets import load_dataset
import pandas as pd

# split[:100] limita o download — leve para Colab Free
print('⏳ Carregando PubMedQA (100 exemplos)...')
pubmedqa = load_dataset(
    'qiaojin/PubMedQA',
    'pqa_labeled',
    split='train[:100]',
    trust_remote_code=True
)
df_pubmed = pd.DataFrame(pubmedqa)
print(f'✅ {len(df_pubmed)} registros carregados')

# Inspecionar
row = df_pubmed.iloc[0]
print('\nPergunta :', row['question'])
print('Decisão  :', row['final_decision'])

## 🏗️ 3. Dados Sintéticos Hospitalares
> 10 protocolos clínicos escritos manualmente — representativos, sem custo computacional.

In [ ]:
SINTETICOS = [
    {
        "instruction": "Qual é o protocolo para manejo inicial de sepse?",
        "input": "",
        "output": (
            "Bundle de 1 hora (Surviving Sepsis Campaign):\n"
            "1. Medir lactato sérico.\n"
            "2. Coletar hemoculturas antes dos antibióticos.\n"
            "3. Iniciar antibiótico de amplo espectro em até 1h.\n"
            "4. Cristaloide 30 mL/kg se hipotensão ou lactato >= 4 mmol/L.\n"
            "5. Norepinefrina se PAM < 65 mmHg após volume.\n"
            "FONTE: Protocolo Sepse 2024. \u26a0\ufe0f Validar com médico assistente."
        )
    },
    {
        "instruction": "Como conduzir dor torácica na emergência?",
        "input": "",
        "output": (
            "1. ECG em até 10 minutos da chegada.\n"
            "2. Troponina T0 e repetir em 1-3h.\n"
            "3. IAMCSST: acionar hemodinâmica imediatamente.\n"
            "4. IAMSSST: anticoagulação + monitorização.\n"
            "5. O2 suplementar apenas se SpO2 < 94%.\n"
            "FONTE: Protocolo Emergência Cardiológica. \u26a0\ufe0f Decisão final do médico."
        )
    },
    {
        "instruction": "Quais os critérios de alta da UTI após cirurgia cardíaca?",
        "input": "",
        "output": (
            "Critérios:\n"
            "- Hemodinâmica estável sem vasopressores por >= 6h.\n"
            "- Extubado com SpO2 >= 94%.\n"
            "- Diurese >= 0,5 mL/kg/h nas últimas 6h.\n"
            "- Ausência de arritmias complexas.\n"
            "- Drenagem torácica < 100 mL/h por 2h consecutivas.\n"
            "- Glasgow >= 14.\n"
            "FONTE: Protocolo UTI Pós-Cardíaco. \u26a0\ufe0f Alta autorizada pelo intensivista."
        )
    },
    {
        "instruction": "Qual antibiótico para pneumonia comunitária sem comorbidades?",
        "input": "",
        "output": (
            "1a escolha: Amoxicilina 1g VO 8/8h por 5-7 dias.\n"
            "Atípicos: Azitromicina 500mg VO 1x/dia por 5 dias.\n"
            "Alergia penicilina: Doxiciclina 100mg VO 12/12h.\n"
            "Internar se CURB-65 >= 2.\n"
            "FONTE: Protocolo Infectologia 2024. \u26a0\ufe0f Prescrição validada pelo médico."
        )
    },
    {
        "instruction": "Como interpretar lesão focal hepática com realce arterial e washout na RM?",
        "input": "Paciente hepatopata crônico, 58 anos. Lesão 2,3 cm, segmento VII.",
        "output": (
            "Padrão sugestivo de CHC — LI-RADS 5.\n"
            "Conduta:\n"
            "1. Reunião multidisciplinar de oncologia hepática.\n"
            "2. Estadiamento: TC tórax + AFP sérica.\n"
            "3. Avaliar função hepática para elegibilidade terapêutica.\n"
            "FONTE: Protocolo Oncologia Hepática 2024. \u26a0\ufe0f Diagnóstico final: médico assistente."
        )
    },
    {
        "instruction": "Qual a conduta para hipoglicemia grave em paciente internado?",
        "input": "",
        "output": (
            "Glicemia < 54 mg/dL com sintomas:\n"
            "1. Via oral possível: 15-20g de carboidrato simples.\n"
            "2. Inconsciente: Glicose 50% 40-60 mL IV em bolus.\n"
            "3. Repetir glicemia em 15 min.\n"
            "4. Manter glicose IV de manutenção se necessário.\n"
            "FONTE: Protocolo Endocrinologia. \u26a0\ufe0f Ajustar esquema insulínico com médico."
        )
    },
    {
        "instruction": "Quando indicar anticoagulação em fibrilação atrial?",
        "input": "",
        "output": (
            "Usar escore CHA2DS2-VASc:\n"
            "- Homem >= 2 pontos: anticoagular.\n"
            "- Mulher >= 3 pontos: anticoagular.\n"
            "Preferir NOAC (rivaroxabana, apixabana) sobre warfarina.\n"
            "FONTE: Diretriz SBC de FA 2023. \u26a0\ufe0f Decisão individualizada pelo cardiologista."
        )
    },
    {
        "instruction": "Como manejar crise hipertensiva sem lesão de órgão-alvo?",
        "input": "PA 220/130 mmHg, paciente assintomático.",
        "output": (
            "Urgência hipertensiva:\n"
            "Objetivo: reduzir PAM em 25% nas primeiras 24h (não abruptamente).\n"
            "1. Captopril 25mg SL ou VO.\n"
            "2. Clonidina 0,1mg VO se necessário.\n"
            "3. Observação mínima 6h.\n"
            "FONTE: Protocolo Emergência HAS. \u26a0\ufe0f UTI se emergência hipertensiva com LOA."
        )
    },
    {
        "instruction": "Quais exames solicitar na avaliação inicial de insuficiência renal aguda?",
        "input": "",
        "output": (
            "Laboratoriais: creatinina, ureia, eletrólitos, gasometria, hemograma.\n"
            "Urina: EAS + sedimento + relação proteína/creatinina.\n"
            "Imagem: USG renal (descartar obstrução).\n"
            "Classificar por KDIGO: estágio 1, 2 ou 3.\n"
            "FONTE: Protocolo Nefrologia. \u26a0\ufe0f Avaliação nefrológica se estágio 2 ou 3."
        )
    },
    {
        "instruction": "Como interpretar troponina elevada?",
        "input": "Troponina I: 0,8 ng/mL (referência < 0,04).",
        "output": (
            "Elevação 20x acima do limite superior — significativa.\n"
            "Diagnóstico diferencial:\n"
            "- Cardíaco: IAM, miocardite, IC descompensada.\n"
            "- Não cardíaco: TEP, sepse, IRA, rabdomiólise.\n"
            "Conduta: ECG seriado + curva de troponina (repetir em 1-3h) + eco.\n"
            "FONTE: Protocolo Emergência Cardiológica. \u26a0\ufe0f Correlação clínica obrigatória."
        )
    }
]

df_sintetico = pd.DataFrame(SINTETICOS)
print(f'✅ {len(df_sintetico)} protocolos sintéticos prontos')

## 🔒 4. Anonimização com Regex
> Leve e eficiente — sem modelos NER.

In [ ]:
import re

def anonimizar(texto: str) -> str:
    if not isinstance(texto, str):
        return texto
    texto = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}', '[CPF]', texto)          # CPF
    texto = re.sub(r'\d{2}[/-]\d{2}[/-]\d{4}', '[DATA]', texto)             # Datas
    texto = re.sub(r'\(?\d{2}\)?[\s-]?\d{4,5}-\d{4}', '[TELEFONE]', texto) # Telefone
    texto = re.sub(r'(?i)(prontu[aá]rio|registro)\s*[#n°]*\s*\d+',          # Prontuário
                   r'\1 [ID]', texto)
    texto = re.sub(r'\d{5}-\d{3}', '[CEP]', texto)                          # CEP
    texto = re.sub(r'(?i)paciente\s+[A-Z][a-z]+\s+[A-Z][a-z]+',            # Nome
                   'Paciente [NOME]', texto)
    return texto

# Teste rápido
teste = "Paciente João Silva, CPF 123.456.789-00, admitido em 10/03/2024. Prontuário #98765."
print('Original   :', teste)
print('Anonimizado:', anonimizar(teste))

In [ ]:
# Aplicar nos dados sintéticos
for col in ['instruction', 'input', 'output']:
    df_sintetico[col] = df_sintetico[col].apply(anonimizar)
print('✅ Anonimização concluída!')

## 🔄 5. Formatação Alpaca JSONL
> Formato padrão esperado pelo fine-tuning com `trl` / `transformers`.

In [ ]:
import json

def pubmed_para_alpaca(row):
    ctx = row['context']
    if isinstance(ctx, dict) and 'contexts' in ctx:
        ctx_str = ' '.join(ctx['contexts'])[:400]
    else:
        ctx_str = str(ctx)[:400]
    return {
        "instruction": row['question'],
        "input": ctx_str,
        "output": f"Conclusão baseada em evidências: {row['final_decision']}."
    }

alpaca_pubmed    = [pubmed_para_alpaca(r) for _, r in df_pubmed.iterrows()]
alpaca_sintetico = df_sintetico.to_dict(orient='records')

dataset_final = alpaca_pubmed + alpaca_sintetico

print(f'Total : {len(dataset_final)} exemplos')
print(f'  PubMedQA  : {len(alpaca_pubmed)}')
print(f'  Sintéticos: {len(alpaca_sintetico)}')
print('\nExemplo sintético:')
print(json.dumps(alpaca_sintetico[0], indent=2, ensure_ascii=False))

## ✅ 6. Validação

In [ ]:
import matplotlib.pyplot as plt

df_final = pd.DataFrame(dataset_final)

print('=== Estatísticas ===')
print(f'Total     : {len(df_final)}')
print(f'Nulos     : {df_final.isnull().sum().sum()}')
print(f'Duplicatas: {df_final.duplicated().sum()}')
df_final.drop_duplicates(inplace=True)

df_final['out_len'] = df_final['output'].str.len()
plt.figure(figsize=(7, 3))
plt.hist(df_final['out_len'], bins=20, color='steelblue', edgecolor='white')
plt.title('Distribuição do tamanho dos outputs')
plt.xlabel('Caracteres'); plt.ylabel('Frequência')
plt.tight_layout(); plt.show()

print(f'Output máx: {df_final["out_len"].max()} chars')
print(f'Output mín: {df_final["out_len"].min()} chars')

## 💾 7. Exportação

In [ ]:
import os
os.makedirs('data', exist_ok=True)

# JSONL para fine-tuning
jsonl_path = 'data/dataset_medico.jsonl'
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for item in df_final.drop(columns=['out_len']).to_dict(orient='records'):
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

# CSV para inspeção
df_final.drop(columns=['out_len']).to_csv('data/dataset_medico.csv', index=False)

print(f'✅ Salvo: {jsonl_path}')
print(f'📊 {len(df_final)} exemplos prontos para fine-tuning')
print('\n🚀 Fase 1 concluída! Próximo: Fase 2 — Fine-tuning com QLoRA.')

In [ ]:
# Download dos arquivos (Colab)
from google.colab import files
files.download('data/dataset_medico.jsonl')
files.download('data/dataset_medico.csv')